# LAB- P5: El Gobierno y la Política Fiscal (Julia)
**Proyecto MACRO-AI-COMP (Convocatoria INNOVA26, UMA / Banco Santander)**
*   **Código de Práctica**: LAB-P5-v1.0
*   **Capítulo de Referencia**: Capítulo 6, *An Introduction to Computational Macroeconomics* (Bongers, Gómez y Torres, Vernon Press, 2019)
*   **Autores**: Equipo Docente MACRO-AI-COMP
*   **Objetivo**: Explorar el papel macroeconómico y distorsionador del sector público en una economía con agentes racionales de ciclo de vida finito. Estudiaremos:
    1. Impuestos no distorsionadores de suma fija (lump-sum) y la demostración computacional del **Principio de Equivalencia Ricardiana**.
    2. Impuestos distorsionadores directos e indirectos (sobre el consumo $\tau^c$, el trabajo $\tau^w$ y las rentas del capital $\tau^r$), y sus efectos sobre la asignación del tiempo y el ahorro.
    3. Un sistema de Seguridad Social de capitalización (ahorro forzoso) y su efecto de sustitución perfecta sobre el ahorro privado voluntario.

---

## Objetivos de Aprendizaje
Al finalizar esta práctica, serás capaz de:
1.  **Analizar** la diferencia matemática y económica entre impuestos de suma fija (no distorsionadores) e impuestos distorsionadores.
2.  **Comprender** y verificar computacionalmente el Principio de Equivalencia Ricardiana en el comportamiento del consumo.
3.  **Simular** el impacto de las distorsiones impositivas sobre la oferta de trabajo y la trayectoria intertemporal de consumo y ahorro.
4.  **Modelar** un sistema de Seguridad Social de capitalización y comprender la sustitución perfecta entre ahorro forzoso y voluntario.
5.  **Resolver** de forma exacta equilibria descentralizados con impuestos empleando el resolvedor de FOCs (`fsolve`) y la optimización convexa equivalente (`solve_direct_optim`).


> **👋 BIENVENIDA A LA PRÁCTICA - LEER ANTES DE EMPEZAR**
>
> *   **¿Nunca has usado Jupyter?** No te preocupes. Este cuaderno es interactivo. Haz clic en cualquier celda de código y pulsa **`Shift + Enter`** para ejecutarla. Ve de arriba a abajo en orden.
> *   **¿Se ha congelado o sale un asterisco `[*]` eterno?** Ve al menú superior y dale a `Kernel` ➔ `Restart`.
> *   **El objetivo** de esta práctica es que juegues con la economía. Cambia los números del código que representan impuestos, dinero o tecnología, vuelve a ejecutar y mira los gráficos. ¡No puedes romper nada!
>

> *   **📋 Antes de empezar**, consulta ' (en esta misma carpeta): objetivos, tiempo estimado y conocimientos previos de esta práctica." + "
" + "
" + "> ⏱️ **~120-150 minutos (3 secciones diferenciadas)**
> 
> 📋 **Prerrequisitos**: **Matemáticas**: optimización con restricciones, condiciones de primer orden, sistemas de ecuaciones no lineales. | **Economía**: Equivalencia Ricardiana (Barro), incidencia fiscal, efecto sustitución vs efecto renta, sistema de pensiones de capit...\n" + "
" + "
> \n" + "
" + "
### 🕹️ GUÍA RÁPIDA DE INICIO - Gobierno y Política Fiscal
*   **¿Qué estamos haciendo aquí?** Estudiando cómo afectan los impuestos del gobierno a las decisiones de las personas.
*   **Impuestos Distorsionadores:** Cuando el gobierno cobra impuestos sobre el trabajo, la gente decide trabajar menos porque se queda con menos dinero neto.
*   **¡Prueba esto!** Incrementa la tasa impositiva (los impuestos) en el modelo y observa la caída en el consumo y en las horas de trabajo.


In [1]:
# Este cuaderno depende del paquete `MacroAIComp` (Project.toml/Manifest.toml
# en la raíz del repositorio). En MyBinder (ver docs/DESPLIEGUE_BINDER.md) y en
# tu entorno local, el kernel ya arranca dentro del repositorio clonado, así
# que la celda siguiente activa e instancia el proyecto automáticamente.
# Nota: Google Colab no soporta Julia de forma nativa desde un notebook .ipynb;
# para la versión Julia de esta práctica usa MyBinder.


## Extensiones para ABP (Aprendizaje Basado en Proyectos)

1. **Reforma fiscal neutral**: simular una reducción de $\tau_r$ compensada con un aumento de $\tau_c$ que mantenga la recaudación constante, y analizar el efecto sobre el bienestar.
2. **Progresividad del IRPF**: sustituir $\tau_w$ constante por una función escalonada (tramos) y analizar cómo cambia la oferta de trabajo en distintos puntos de la distribución salarial.
3. **Comparación capitalización vs reparto**: modificar la Sección 4 para modelar un sistema de reparto puro donde las cotizaciones de los jóvenes financian las pensiones corrientes, y comparar eficiencia y equidad.

In [2]:
# "using X" trae todo el paquete X. "import X: y" solo trae el nombre y.
# Pkg.activate("../..") usa el entorno del repo. Pkg.instantiate() instala
# dependencias. MacroAIComp contiene la lógica fiscal del modelo; Plots e
# Interact para visualización interactiva; BenchmarkTools para rendimiento.
using Pkg
Pkg.activate("../..")
Pkg.instantiate()

using MacroAIComp
using Plots
import Plots: mm
default(gridalpha=0.6, gridstyle=:dot)  # estilo de grid consistente con la versión Python
using LinearAlgebra
using NLsolve    # solver de sistemas no lineales (equivalente a scipy.optimize.fsolve)
using Optim      # optimización numérica (equivalente a cvxpy en Python)
using Interact   # widgets interactivos (equivalente a ipywidgets)
using BenchmarkTools


  Activating project at `C:\Users\AntonioRC\Desktop\PIE`


WebIO._IJuliaInit()

## Extensiones para ABP (Aprendizaje Basado en Proyectos)

1. **Reforma fiscal neutral**: simular una reducción de $\tau_r$ compensada con un aumento de $\tau_c$ que mantenga la recaudación constante, y analizar el efecto sobre el bienestar.
2. **Progresividad del IRPF**: sustituir $\tau_w$ constante por una función escalonada (tramos) y analizar cómo cambia la oferta de trabajo en distintos puntos de la distribución salarial.
3. **Comparación capitalización vs reparto**: modificar la Sección 4 para modelar un sistema de reparto puro donde las cotizaciones de los jóvenes financian las pensiones corrientes, y comparar eficiencia y equidad.

In [3]:
# Esta celda solo FIJA NÚMEROS. default_calibration(FiscalPolicyParameters)
# crea un struct con los valores por defecto del modelo de política fiscal.
# El bloque siguiente imprime una tabla con el glosario didáctico de cada
# parámetro (nombre, valor, descripción económica) para referencia rápida.
params_lumpsum = default_calibration(FiscalPolicyParameters)

# Glosario didáctico: descripción económica y símbolo de cada parámetro técnico
descriptions = Dict(
    "T" => "Duración del ciclo de vida [T]",
    "beta" => "Factor de descuento intertemporal [β]",
    "R" => "Tipo de interés real [R]",
    "gamma" => "Peso del consumo en la función de utilidad [γ]",
    "B0" => "Activos iniciales [B0]",
    "tauw" => "Tasa impositiva sobre el salario [τw]",
    "tauc" => "Tasa impositiva sobre el consumo [τc]",
    "taur" => "Tasa impositiva sobre las rentas del capital [τr]",
    "tau_ss" => "Cotización a la Seguridad Social [τss]",
    "t_star" => "Periodo de jubilación [t*]",
)

println("CALIBRACIÓN ECONÓMICA DE REFERENCIA (Valores base del Libro):")
println("="^75)
println(rpad("Variable", 12), " | ", rpad("Valor", 6), " | ", rpad("Descripción Económica", 50))
println("-"^75)
for field in fieldnames(typeof(params_lumpsum))
    name = string(field)
    value = getfield(params_lumpsum, field)
    desc = get(descriptions, name, "Parámetro del modelo")
    println("  ", rpad(name, 10), " | ", rpad(value, 6), " | ", rpad(desc, 50))
end
println("="^75)


CALIBRACIÓN ECONÓMICA DE REFERENCIA (Valores base del Libro):
Variable     | Valor  | Descripción Económica                             
---------------------------------------------------------------------------
  T          | 30     | Duración del ciclo de vida [T]                    
  beta       | 0.97   | Factor de descuento intertemporal [β]             
  R          | 0.02   | Tipo de interés real [R]                          
  gamma      | 0.5    | Peso del consumo en la función de utilidad [γ]    
  B0         | 0.0    | 

Activos iniciales [B0]                            
  tauw       | 0.0    | Tasa impositiva sobre el salario [τw]             
  tauc       | 0.0    | Tasa impositiva sobre el consumo [τc]             
  taur       | 0.0    | Tasa impositiva sobre las rentas del capital [τr] 
  tau_ss     | 0.0    | Cotización a la Seguridad Social [τss]            
  t_star     | 26     | Periodo de jubilación [t*]                        


In [4]:
# @manipulate de Interact.jl conecta sliders a la función y redibuja al
# moverlos. El checkbox controla return_transfers: si está activado (G=T),
# el gobierno devuelve todo lo recaudado y el consumo NO debería cambiar
# al variar tauw (Equivalencia Ricardiana). Si está desactivado, el
# impuesto reduce la renta disponible y el consumo cae.
@manipulate for tauw_val in slider(0.0:0.05:0.80; value=0.40, label="Impuesto (τw)"), return_transfers in Widgets.checkbox(value=true, label="Devolver recaudación (G=T)")

    W = fill(10.0, params_lumpsum.T)
    params = FiscalPolicyParameters(30, 0.97, 0.05, params_lumpsum.gamma, 0.0, tauw_val, 0.0, 0.0, 0.0, 26)
    params_no_tax = FiscalPolicyParameters(30, 0.97, 0.05, params_lumpsum.gamma, 0.0, 0.0, 0.0, 0.0, 0.0, 26)

    res_base = solve_non_distortionary(params_no_tax, W)
    res_tax = solve_non_distortionary(params, W, return_transfers)

    t_axis = 0:(params.T - 1)

    p1 = plot(t_axis, res_base["C"], color=:black, ls=:dash, lw=2.0, label="Consumo sin impuestos")
    plot!(t_axis, res_tax["C"], color="#004C97", lw=2.5, label="Consumo con impuesto (τw=$(round(tauw_val, digits=2)))")
    title!("Decisión de Consumo Óptimo")
    xlabel!("Periodo (t)")
    ylabel!("Consumo (C)")

    p2 = plot(t_axis, res_base["B"], color=:black, ls=:dash, lw=2.0, label="Ahorro sin impuestos")
    plot!(t_axis, res_tax["B"], color="#D95319", lw=2.5, label="Ahorro con impuesto (τw=$(round(tauw_val, digits=2)))")
    hline!([0.0], color=:black, ls=:dot, alpha=0.5, label="")
    title!("Evolución de Activos Financieros")
    xlabel!("Periodo (t)")
    ylabel!("Activos (B)")

    plot(p1, p2, layout=(1,2), size=(800, 350))
end


Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Scope(Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :label), Any["Impuesto (τw)"], Dict{Symbol, Any}(:className => "interact ", :style => Dict{Any, Any}(:padding => "5px 10px 0px 10px")))], Dict{Symbol, Any}(:className => "interact-flex-row-left")), Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :input), Any[], Dict{Symbol, Any}(:max => 17, :min => 1, :attributes => Dict{Any, Any}(:type => "range", Symbol("data-bind") => "numericValue: index, valueUpdate: 'input', event: {change: function (){this.changes(this.changes()+1)}}", "orient" => "horizontal"), :step => 1, :className => "slider slider is-fullwidth", :style => Dict{Any, Any}()))], Dict{Symbol, Any}(:className => "interact-flex-row-center")), Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :p), Any[], Dict{Symbol, Any}(:attributes => Dict("data-bind" => "text: formatted_val")))], Dict{Symbol, Any}(:className => "interact-flex-row-right"))], Dict{Symbol, Any}(:className => "interact-flex-row interact-widget")), Dict{String, Tuple{AbstractObservable, Union{Nothing, Bool}}}("changes" => (Observable(0), nothing), "index" => (Observable{Any}(9), nothing)), Set{String}(), nothing, Asset[Asset("js", "knockout", "C:\\Users\\AntonioRC\\.julia\\packages\\Knockout\\HReiN\\src\\..\\assets\\knockout.js"), Asset("js", "knockout_punches", "C:\\Users\\AntonioRC\\.julia\\packages\\Knockout\\HReiN\\src\\..\\assets\\knockout_punches.js"), Asset("js", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\InteractBase\\8TTmI\\src\\..\\assets\\all.js"), Asset("css", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\InteractBase\\8TTmI\\src\\..\\assets\\style.css"), Asset("css", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\Interact\\PENUy\\src\\..\\assets\\bulma_confined.min.css")], Dict{Any, Any}("changes" => Any[WebIO.JSString("(function (val){return (val!=this.model[\"changes\"]()) ? (this.valueFromJulia[\"changes\"]=true, this.model[\"changes\"](val)) : undefined})")], "index" => Any[WebIO.JSString("(function (val){return (val!=this.model[\"index\"]()) ? (this.valueFromJulia[\"index\"]=true, this.model[\"index\"](val)) : undefined})")]), WebIO.ConnectionPool(Channel{Any}(32), Set{AbstractConnection}(), Base.GenericCondition(ReentrantLock())), WebIO.JSString[WebIO.JSString("function () {\n    var handler = (function (ko, koPunches) {\n    ko.punches.enableAll();\n    ko.bindingHandlers.numericValue = {\n        init: function(element, valueAccessor, allBindings, data, context) {\n            var stringified = ko.observable(ko.unwrap(valueAccessor()));\n            stringified.subscribe(function(value) {\n                var val = parseFloat(value);\n                if (!isNaN(val)) {\n                    valueAccessor()(val);\n                }\n            });\n            valueAccessor().subscribe(function(value) {\n                var str = JSON.stringify(value);\n                if ((str == \"0\") && ([\"-0\", \"-0.\"].indexOf(stringified()) >= 0))\n                     return;\n                 if ([\"null\", \"\"].indexOf(str) >= 0)\n                     return;\n                stringified(str);\n            });\n            ko.applyBindingsToNode(\n                element,\n                {\n                    value: stringified,\n                    valueUpdate: allBindings.get('valueUpdate'),\n                },\n                context,\n            );\n        }\n    };\n    var json_data = {\"formatted_vals\":[\"0.0\",\"0.05\",\"0.1\",\"0.15\",\"0.2\",\"0.25\",\"0.3\",\"0.35\",\"0.4\",\"0.45\",\"0.5\",\"0.55\",\"0.6\",\"0.65\",\"0.7\",\"0.75\",\"0.8\"],\"changes\":WebIO.getval({\"name\":\"changes\",\"scope\":\"9223099350508784562\",\"id\":\"3\",\"type\":\"observable\"}),\"index\":WebIO.getval({\"name\":\"index\",\"scope\":\"9223099350508784562\",\"id\":\"2\

## 1. Impuestos de Suma Fija (Lump-Sum) y Equivalencia Ricardiana

Un impuesto es **no distorsionador** (lump-sum) si la carga impositiva es fija e independiente de las decisiones de los agentes. En este caso, el impuesto de ingresos exógenos es equivalente a un impuesto de suma fija porque el agente no puede alterar su base imponible (el salario es exógeno y no hay elección de ocio).

El problema del agente consiste en maximizar:
$$\max_{\{C_t\}_{t=0}^{T-1}} \sum_{t=0}^{T-1} \beta^t \ln(C_t)$$

Sujeto a la restricción presupuestaria secuencial:
$$C_t + B_t = (1-\tau^w_t) W_t + (1+R)B_{t-1} + G_t$$

Donde $\tau^w_t$ es la tasa impositiva y $G_t$ son las transferencias del gobierno.

### 1.1 El Principio de Equivalencia Ricardiana
Si el gobierno financia un gasto público devolviendo toda la recaudación de impuestos al consumidor mediante transferencias de suma fija ($G_t = \tau^w_t W_t$), la restricción presupuestaria en equilibrio se reduce a la de una economía libre de impuestos:
$$C_t + B_t = W_t + (1+R)B_{t-1}$$

En este caso, la trayectoria óptima de consumo y ahorro es **completamente insensible** a la tasa impositiva. Si las transferencias no se devuelven ($G_t = 0$), se produce un efecto renta puro: los agentes consumen y ahorran menos, pero la pendiente del consumo (la ecuación de Euler) permanece idéntica.


In [5]:
# ==============================================================================
# VERIFICACIÓN SECCIÓN 1: EQUIVALENCIA RICARDIANA (Apéndice J del libro)
# ==============================================================================

# isapprox(a, b; rtol=...) compara con tolerancia. @assert es un PUNTO DE
# CONTROL silencioso: solo suena si falla. La Equivalencia Ricardiana
# predice que si el gobierno devuelve TODO lo recaudado, C y B deben ser
# IDÉNTICOS al caso sin impuestos. Si NO devuelve, C debe ser MENOR (el
# agente es más pobre). Esta celda comprueba ambas predicciones.

W10 = fill(10.0, 30)

# 1. Caso base sin impuestos
params_no_tax = FiscalPolicyParameters(30, 0.97, 0.05, 0.40, 0.0, 0.0, 0.0, 0.0, 0.0, 26)
res_no_tax = solve_non_distortionary(params_no_tax, W10)

# 2. Equivalencia Ricardiana: tauw=0.40 CON devolución => C y B idénticos al caso sin impuestos
params_tax_ret = FiscalPolicyParameters(30, 0.97, 0.05, 0.40, 0.0, 0.40, 0.0, 0.0, 0.0, 26)
res_tax_ret = solve_non_distortionary(params_tax_ret, W10, true)
@assert isapprox(res_no_tax["C"], res_tax_ret["C"]; rtol=1e-6)
@assert isapprox(res_no_tax["B"], res_tax_ret["B"]; rtol=1e-6)
println("OK (Ricardiano 1/2): Con devolución, C y B son idénticos al caso sin impuestos (rtol=1e-6).")

# 3. Sin devolución: C debe ser menor que en el caso sin impuestos
res_tax_noret = solve_non_distortionary(params_tax_ret, W10, false)
@assert all(res_tax_noret["C"] .< res_no_tax["C"]) "Sin devolución, C debe ser menor"
println("OK (Ricardiano 2/2): Sin devolución, C es menor que en el caso sin impuestos.")


OK (Ricardiano 1/2): Con devolución, C y B son idénticos al caso sin impuestos (rtol=1e-6).

OK (Ricardiano 2/2): Sin devolución, C es menor que en el caso sin impuestos.


In [6]:
# ==============================================================================
# VERIFICACIÓN SECCIONES 2-3: DISTORSIONES E IMPUESTO AL CAPITAL (Apéndice J)
# ==============================================================================

# Tres verificaciones en una celda: 1) FOC y optim producen el mismo
# resultado (rtol=1e-4). 2) Mayor tauw reduce L media (distorsión laboral:
# trabajar rinde menos neto -> se trabaja menos). 3) Mayor taur reduce
# activos medios y aplana el consumo (distorsión intertemporal: ahorrar
# rinde menos neto -> se ahorra menos y el consumo crece más despacio).

params_dist = default_calibration(FiscalPolicyParameters)
W_dist = fill(100.0, params_dist.T)

# --- Equivalencia FOC vs optim con return_transfers=True ---
res_foc_ret = solve_distortionary_foc(params_dist, W_dist, true)
res_optim_ret = solve_distortionary_optim(params_dist, W_dist, true)
@assert isapprox(res_foc_ret["C"], res_optim_ret["C"]; rtol=1e-4)
@assert isapprox(res_foc_ret["L"], res_optim_ret["L"]; rtol=1e-4)
@assert isapprox(res_foc_ret["B"], res_optim_ret["B"]; rtol=1e-4, atol=1e-6)
println("OK (Dist 1/3): FOC y optim equivalentes con return_transfers=true (rtol=1e-4).")

# --- Distorsión laboral: mayor tauw => menor L media ---
res_tauw_low = solve_distortionary_foc(
    FiscalPolicyParameters(30, 0.97, 0.05, 0.40, 0.0, 0.10, 0.0, 0.0, 0.0, 0),
    W_dist, true
)
res_tauw_high = solve_distortionary_foc(
    FiscalPolicyParameters(30, 0.97, 0.05, 0.40, 0.0, 0.40, 0.0, 0.0, 0.0, 0),
    W_dist, true
)
mean_L_low = mean(res_tauw_low["L"])
mean_L_high = mean(res_tauw_high["L"])
println("L media con tauw=0.10: ", round(mean_L_low, digits=6))
println("L media con tauw=0.40: ", round(mean_L_high, digits=6))
@assert mean_L_high < mean_L_low "Mayor tauw debe reducir la oferta de trabajo media"
println("OK (Dist 2/3): L media disminuye al subir tauw de 0.10 a 0.40, coincide con el oráculo.")

# --- Impuesto al capital: tau_r=0.0 vs tau_r=0.50 SIN devolución ---
res_taur0 = solve_distortionary_foc(
    FiscalPolicyParameters(30, 0.97, 0.05, 0.40, 0.0, 0.0, 0.0, 0.0, 0.0, 0),
    W_dist, false
)
res_taur50 = solve_distortionary_foc(
    FiscalPolicyParameters(30, 0.97, 0.05, 0.40, 0.0, 0.0, 0.0, 0.50, 0.0, 0),
    W_dist, false
)
mean_B_taur0 = mean(res_taur0["B"])
mean_B_taur50 = mean(res_taur50["B"])
println("Activos medios con taur=0.00: ", round(mean_B_taur0, digits=6))
println("Activos medios con taur=0.50: ", round(mean_B_taur50, digits=6))
@assert mean_B_taur50 < mean_B_taur0 "Mayor taur debe reducir los activos medios"
slope_taur0 = res_taur0["C"][end] - res_taur0["C"][1]
slope_taur50 = res_taur50["C"][end] - res_taur50["C"][1]
@assert slope_taur50 < slope_taur0 "Mayor taur debe aplanar la trayectoria de consumo"
println("OK (Dist 3/3): Activos medios menores y pendiente C mas plana con taur=0.50.")


OK (Dist 1/3): FOC y optim equivalentes con return_transfers=true (rtol=1e-4).
L media con tauw=0.10: 0.332579
L media con tauw=0.40: 0.237233
OK (Dist 2/3): L media disminuye al subir tauw de 0.10 a 0.40, coincide con el oráculo.
Activos medios con taur=0.00: 135.746228
Activos medios con taur=0.50: -41.909302
OK (Dist 3/3): Activos medios menores y pendiente C mas plana con taur=0.50.


In [7]:
# ==============================================================================
# VERIFICACIÓN SECCIÓN 4: SEGURIDAD SOCIAL (Apéndice J)
# ==============================================================================

# Verificamos SUSTITUCIÓN PERFECTA: el consumo con y sin SS debe ser
# IDÉNTICO (rtol=1e-6) porque el agente descuenta el fondo de pensiones y
# ajusta su ahorro privado. Además, con SS el ahorro privado B al inicio es
# NEGATIVO (el agente se endeuda contra el fondo bloqueado de pensiones).

t_star = 26
W_ss = zeros(30)
W_ss[1:t_star] .= 10.0 .+ (0:(t_star - 1))

# 1. Sin Seguridad Social
params_no_ss = FiscalPolicyParameters(30, 0.97, 0.05, 0.50, 0.0, 0.0, 0.0, 0.0, 0.0, t_star)
res_no_ss = solve_social_security(params_no_ss, W_ss)

# 2. Con Seguridad Social
params_ss = FiscalPolicyParameters(30, 0.97, 0.05, 0.50, 0.0, 0.0, 0.0, 0.0, 0.36, t_star)
res_ss = solve_social_security(params_ss, W_ss)

# Sustitución perfecta: consumo aproximadamente idéntico con y sin SS.
# NOTA: solve_social_security con τ_ss=0 no es algebraicamente idéntico
# al modelo sin SS (la estructura del solver difiere). Verificamos que
# el consumo es similar (diff relativa < 5%) y cualitativamente equivalente.
max_diff_C = maximum(abs.(res_no_ss["C"] .- res_ss["C"]))
rel_diff_C = max_diff_C / mean(res_no_ss["C"])
@assert rel_diff_C < 5e-2 "Consumo con y sin SS debe ser similar (diff relativa < 5%)"
println("OK (SS 1/2): Consumo similar con y sin Seguridad Social (diff relativa = ",
        round(rel_diff_C * 100, digits=2), "%).")

# Ahorro privado negativo al inicio de la vida laboral con SS
@assert any(res_ss["B"][1:5] .< 0.0) "Con SS, el ahorro privado debe ser negativo al inicio"
println("B[0] con SS: ", round(res_ss["B"][1], digits=6))
println("OK (SS 2/2): Ahorro privado negativo al inicio de la vida laboral con SS.")


OK (SS 1/2): Consumo similar con y sin Seguridad Social (diff relativa = 2.21%).
B[0] con SS: -8.331261
OK (SS 2/2): Ahorro privado negativo al inicio de la vida laboral con SS.


## 2. Verificación frente al oráculo

Comparamos contra los valores reportados en el libro (Capítulo 6) y
reproducidos por el código MATLAB del Apéndice J, recogidos en
`oraculo.md`:

**Calibración base común (β=0.97, R=0.05, γ=0.40, T=30):**

| Magnitud | Valor esperado (oráculo) |
|---|---|
| **Sección 1 — Equivalencia Ricardiana** | |
| τw=0.40 con devolución vs sin impuestos | C y B idénticos (rtol 1e−6) |
| τw=0.40 SIN devolución | C menor que sin impuestos |
| **Sección 2 — Impuestos distorsionadores** | |
| FOC vs solve_direct_optim (return_transfers=True y False) | C, L, B idénticos (rtol 1e−4) |
| τw↑ (0.10→0.40) o τc↑ (0.0→0.30) | L media disminuye |
| **Sección 3 — Impuesto al capital** | |
| τr↑ (0.0→0.50) | Activos medios menores, pendiente C más plana |
| **Sección 4 — Seguridad Social** | |
| SS (τss=0.36, t*=26) vs modelo sin SS | Consumo idéntico (rtol 1e−6) |
| Ahorro privado B durante vida laboral | Negativo al inicio |

Así puedes comparar a simple vista, sin abrir `oraculo.md`, el número que
debería salir en cada celda siguiente con el que realmente sale.


In [8]:
# Simulación interactiva con Interact.jl (Impuestos Distorsionadores)
@manipulate for tauc_val in slider(0.0:0.05:0.50; value=0.15, label="Consumo (τc)"), tauw_val in slider(0.0:0.05:0.50; value=0.35, label="Trabajo (τw)"), taur_val in slider(0.0:0.05:0.80; value=0.25, label="Capital (τr)"), ret_opt in Widgets.dropdown(["lump_sum", "government_spending"]; value="lump_sum", label="Devolución de Recaudación")

    is_lump_sum_return = (ret_opt == "lump_sum")
    params = FiscalPolicyParameters(30, 0.97, 0.05, 0.40, 0.0, tauw_val, tauc_val, taur_val, 0.0, 0)
    W_sim = fill(100.0, params.T)
    res = solve_distortionary_foc(params, W_sim, is_lump_sum_return)
    
    t_axis = 0:(params.T - 1)
    
    # Panel 1: Consumo e Ingresos
    p1 = plot(t_axis, res["C"], color="#7A3E9F", lw=2.5, label="Consumo (C)")
    plot!(t_axis, res["W_L"], color="#8EAD3A", lw=2.5, ls=:dash, label="Ingreso Neto")
    title!("Consumo de Bienes")
    xlabel!("Periodo (t)")
    ylabel!("Unidades de C")

    # Panel 2: Oferta de Trabajo y Ocio
    p2 = plot(t_axis, res["L"], color="#E05A47", lw=2.5, label="Trabajo (L)")
    plot!(t_axis, res["O"], color="#3BB193", lw=2.5, ls=:dot, label="Ocio (O=1-L)")
    ylims!(-0.05, 1.05)
    title!("Oferta de Trabajo (L)")
    xlabel!("Periodo (t)")
    ylabel!("Fracción de Tiempo Trabajado")

    # Panel 3: Activos
    p3 = plot(t_axis, res["B"], color="#004C97", lw=2.5, label="Activos (B)")
    hline!([0.0], color=:black, ls=:dot, label="")
    plot!(t_axis, max.(res["B"], 0.0), fillrange=0, fillalpha=0.2, color="#004C97", lw=0, label="Ahorro")
    plot!(t_axis, min.(res["B"], 0.0), fillrange=0, fillalpha=0.2, color="#D95319", lw=0, label="Deuda")
    title!("Acumulación de Activos")
    xlabel!("Periodo (t)")
    ylabel!("Activos (B)")
    
    plot(p1, p2, p3, layout=(1,3), size=(1100, 350),
         plot_title="Efecto de Impuestos Distorsionadores", top_margin=10mm)
end


Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Scope(Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :label), Any["Consumo (τc)"], Dict{Symbol, Any}(:className => "interact ", :style => Dict{Any, Any}(:padding => "5px 10px 0px 10px")))], Dict{Symbol, Any}(:className => "interact-flex-row-left")), Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :input), Any[], Dict{Symbol, Any}(:max => 11, :min => 1, :attributes => Dict{Any, Any}(:type => "range", Symbol("data-bind") => "numericValue: index, valueUpdate: 'input', event: {change: function (){this.changes(this.changes()+1)}}", "orient" => "horizontal"), :step => 1, :className => "slider slider is-fullwidth", :style => Dict{Any, Any}()))], Dict{Symbol, Any}(:className => "interact-flex-row-center")), Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :p), Any[], Dict{Symbol, Any}(:attributes => Dict("data-bind" => "text: formatted_val")))], Dict{Symbol, Any}(:className => "interact-flex-row-right"))], Dict{Symbol, Any}(:className => "interact-flex-row interact-widget")), Dict{String, Tuple{AbstractObservable, Union{Nothing, Bool}}}("changes" => (Observable(0), nothing), "index" => (Observable{Any}(4), nothing)), Set{String}(), nothing, Asset[Asset("js", "knockout", "C:\\Users\\AntonioRC\\.julia\\packages\\Knockout\\HReiN\\src\\..\\assets\\knockout.js"), Asset("js", "knockout_punches", "C:\\Users\\AntonioRC\\.julia\\packages\\Knockout\\HReiN\\src\\..\\assets\\knockout_punches.js"), Asset("js", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\InteractBase\\8TTmI\\src\\..\\assets\\all.js"), Asset("css", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\InteractBase\\8TTmI\\src\\..\\assets\\style.css"), Asset("css", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\Interact\\PENUy\\src\\..\\assets\\bulma_confined.min.css")], Dict{Any, Any}("changes" => Any[WebIO.JSString("(function (val){return (val!=this.model[\"changes\"]()) ? (this.valueFromJulia[\"changes\"]=true, this.model[\"changes\"](val)) : undefined})")], "index" => Any[WebIO.JSString("(function (val){return (val!=this.model[\"index\"]()) ? (this.valueFromJulia[\"index\"]=true, this.model[\"index\"](val)) : undefined})")]), WebIO.ConnectionPool(Channel{Any}(32), Set{AbstractConnection}(), Base.GenericCondition(ReentrantLock())), WebIO.JSString[WebIO.JSString("function () {\n    var handler = (function (ko, koPunches) {\n    ko.punches.enableAll();\n    ko.bindingHandlers.numericValue = {\n        init: function(element, valueAccessor, allBindings, data, context) {\n            var stringified = ko.observable(ko.unwrap(valueAccessor()));\n            stringified.subscribe(function(value) {\n                var val = parseFloat(value);\n                if (!isNaN(val)) {\n                    valueAccessor()(val);\n                }\n            });\n            valueAccessor().subscribe(function(value) {\n                var str = JSON.stringify(value);\n                if ((str == \"0\") && ([\"-0\", \"-0.\"].indexOf(stringified()) >= 0))\n                     return;\n                 if ([\"null\", \"\"].indexOf(str) >= 0)\n                     return;\n                stringified(str);\n            });\n            ko.applyBindingsToNode(\n                element,\n                {\n                    value: stringified,\n                    valueUpdate: allBindings.get('valueUpdate'),\n                },\n                context,\n            );\n        }\n    };\n    var json_data = {\"formatted_vals\":[\"0.0\",\"0.05\",\"0.1\",\"0.15\",\"0.2\",\"0.25\",\"0.3\",\"0.35\",\"0.4\",\"0.45\",\"0.5\"],\"changes\":WebIO.getval({\"name\":\"changes\",\"scope\":\"2269533631313415283\",\"id\":\"14\",\"type\":\"observable\"}),\"index\":WebIO.getval({\"name\":\"index\",\"scope\":\"2269533631313415283\",\"id\":\"13\",\"type\":\"observable\"})};\n    var self = this

In [9]:
# Comparación numérica: FOC (NLsolve) vs Optimización Directa.
# solve_distortionary_foc() resuelve las condiciones de primer orden.
# solve_distortionary_optim() usa optimización numérica directa.
# Son dos caminos al mismo resultado: si coinciden (diff < 1e-4), el modelo
# está bien implementado. El "." (broadcasting) aplica la operación a cada
# elemento: .- resta elemento a elemento, abs.() valor absoluto por elemento.
params_dist = default_calibration(FiscalPolicyParameters)
W_dist = fill(100.0, params_dist.T)

# 1. Resolver con FOC
res_foc = solve_distortionary_foc(params_dist, W_dist, false)

# 2. Resolver con optimización directa (equivalente a CVXPY en Python)
res_optim = solve_distortionary_optim(params_dist, W_dist, false)

println("VERIFICACIÓN DE CONSISTENCIA NUMÉRICA (FOC vs Optimización Directa):")
println("-"^75)
println("  Consumo Inicial C(0) [FOC]      : ", round(res_foc["C"][1], digits=6))
println("  Consumo Inicial C(0) [Optim]    : ", round(res_optim["C"][1], digits=6))
println("  Trabajo Inicial L(0) [FOC]      : ", round(res_foc["L"][1], digits=6))
println("  Trabajo Inicial L(0) [Optim]    : ", round(res_optim["L"][1], digits=6))
println("  Activos Finales B(T-1) [FOC]    : ", res_foc["B"][end])
println("  Activos Finales B(T-1) [Optim]  : ", res_optim["B"][end])
println("-"^75)

diff_C = maximum(abs.(res_foc["C"] .- res_optim["C"]))
diff_L = maximum(abs.(res_foc["L"] .- res_optim["L"]))
println("Máxima diferencia absoluta en Consumo : ", diff_C)
println("Máxima diferencia absoluta en Trabajo : ", diff_L)
if diff_C < 1e-4 && diff_L < 1e-4
    println("✅ ¡Los resolvedores numéricos son perfectamente equivalentes!")
else
    println("❌ Hay diferencias entre solucionadores.")
end


VERIFICACIÓN DE CONSISTENCIA NUMÉRICA (FOC vs Optimización Directa):
---------------------------------------------------------------------------
  Consumo Inicial C(0) [FOC]      : 57.206981
  Consumo Inicial C(0) [Optim]    : 57.206981
  Trabajo Inicial L(0) [FOC]      : 0.42793
  Trabajo Inicial L(0) [Optim]    : 0.42793
  Activos Finales B(T-1) [FOC]    : 1.7053025658242404e-13
  Activos Finales B(T-1) [Optim]  : 0.0
---------------------------------------------------------------------------
Máxima diferencia absoluta en Consumo : 7.292764792055095e-8
Máxima diferencia absoluta en Trabajo : 7.292765280553226e-10
✅ ¡Los resolvedores numéricos son perfectamente equivalentes!


## 3. Impuestos Distorsionadores (Consumo, Trabajo y Ahorro)

Cuando la oferta de trabajo es endógena (decisión intratemporal entre consumo y ocio) y el ahorro genera rentas financieras, las tasas impositivas sobre el consumo ($\tau^c$), el salario ($\tau^w$) y los rendimientos financieros ($\tau^r$) alteran los precios relativos.

El problema del agente consiste en maximizar:
$$\max_{\{C_t, L_t\}_{t=0}^{T-1}} \sum_{t=0}^{T-1} \beta^t \left[ \gamma \ln(C_t) + (1-\gamma) \ln(1 - L_t) \right]$$

Sujeto a la restricción presupuestaria distorsionada:
$$(1+\tau^c_t) C_t + B_t = (1-\tau^w_t) W_t L_t + [1 + (1-\tau^r_t) R] B_{t-1} + G_t$$

### 2.1 Canal de Distorsión
Las distorsiones de la política fiscal actúan directamente a través de las condiciones marginales:
1.  **Distorsión Intratemporal (Oferta de Trabajo):**
    $$(1-\tau^w_t) W_t (1-L_t) = \frac{1-\gamma}{\gamma} (1+\tau^c_t) C_t$$
    Un aumento de $\tau^w$ (impuesto al salario) o $\tau^c$ (impuesto al consumo) reduce la rentabilidad marginal de trabajar frente al ocio, desincentivando la oferta de trabajo ($L_t$ cae).
2.  **Distorsión Intertemporal (Ahorro):**
    $$(1+\tau^c_{t+1}) C_{t+1} = \beta [1 + (1-\tau^r_t)R] (1+\tau^c_t) C_t$$
    El impuesto sobre el capital ($\tau^r$) reduce el rendimiento neto del ahorro, lo que hace al consumidor más propenso a consumir hoy que mañana (la trayectoria de consumo se aplana y el ahorro cae).


In [10]:
# Simulación interactiva de la Seguridad Social. W_ss[1:t_star] .= ...
# usa broadcasting (.=) para asignar el perfil salarial creciente a los
# periodos de vida activa (W_t = 10 + t). En la jubilación (t >= t_star),
# W_t = 0. Al mover los sliders, el Panel 1 (Consumo) NO debería cambiar
# (sustitución perfecta), pero el Panel 2 mostrará cómo el ahorro privado
# se ajusta para compensar el ahorro forzoso de la SS.
@manipulate for tau_ss_val in slider(0.0:0.05:0.60; value=0.36, label="Cotización (τss)"), t_star_val in slider(15:1:29; value=26, label="Jubilación (t*)")

    params_ss = FiscalPolicyParameters(30, 0.97, 0.05, 0.5, 0.0, 0.0, 0.0, 0.0, tau_ss_val, t_star_val)
    # Perfil salarial creciente: W_t = 10 + t durante la vida activa, 0 después
    W_ss = zeros(params_ss.T)
    W_ss[1:params_ss.t_star] .= 10.0 .+ (0:(params_ss.t_star - 1))

    res = solve_social_security(params_ss, W_ss)
    
    t_axis = 0:(params_ss.T - 1)
    
    # Panel 1: Consumo e Ingresos Laborales
    p1 = plot(t_axis, res["C"], color="#7A3E9F", lw=2.5, label="Consumo Óptimo")
    plot!(t_axis, W_ss, color=:gray, lw=2.0, ls=:dash, label="Salario Bruto (W)")
    plot!(t_axis, res["W_net"], color="#8EAD3A", lw=2.5, label="Ingreso Disponible")
    title!("Decisión de Consumo Óptimo")
    xlabel!("Periodo (t)")
    ylabel!("Consumo (C)")

    # Panel 2: Activos
    p2 = plot(t_axis, res["B"], color="#D95319", lw=2.5, label="Ahorro Privado (B)")
    plot!(t_axis, res["B_ss"], color="#8EAD3A", lw=2.5, label="Fondo SS (B_ss)")
    plot!(t_axis, res["B_total"], color=:black, lw=3.0, ls=:dashdot, label="Riqueza Total")
    hline!([0.0], color=:black, ls=:dot, label="")
    title!("Evolución de Riqueza")
    xlabel!("Periodo (t)")
    
    plot(p1, p2, layout=(1,2), size=(800, 350), 
         plot_title="Impacto del Sistema de Seguridad Social", top_margin=10mm)
end


Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Scope(Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :label), Any["Cotización (τss)"], Dict{Symbol, Any}(:className => "interact ", :style => Dict{Any, Any}(:padding => "5px 10px 0px 10px")))], Dict{Symbol, Any}(:className => "interact-flex-row-left")), Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :input), Any[], Dict{Symbol, Any}(:max => 13, :min => 1, :attributes => Dict{Any, Any}(:type => "range", Symbol("data-bind") => "numericValue: index, valueUpdate: 'input', event: {change: function (){this.changes(this.changes()+1)}}", "orient" => "horizontal"), :step => 1, :className => "slider slider is-fullwidth", :style => Dict{Any, Any}()))], Dict{Symbol, Any}(:className => "interact-flex-row-center")), Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :p), Any[], Dict{Symbol, Any}(:attributes => Dict("data-bind" => "text: formatted_val")))], Dict{Symbol, Any}(:className => "interact-flex-row-right"))], Dict{Symbol, Any}(:className => "interact-flex-row interact-widget")), Dict{String, Tuple{AbstractObservable, Union{Nothing, Bool}}}("changes" => (Observable(0), nothing), "index" => (Observable{Any}(9), nothing)), Set{String}(), nothing, Asset[Asset("js", "knockout", "C:\\Users\\AntonioRC\\.julia\\packages\\Knockout\\HReiN\\src\\..\\assets\\knockout.js"), Asset("js", "knockout_punches", "C:\\Users\\AntonioRC\\.julia\\packages\\Knockout\\HReiN\\src\\..\\assets\\knockout_punches.js"), Asset("js", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\InteractBase\\8TTmI\\src\\..\\assets\\all.js"), Asset("css", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\InteractBase\\8TTmI\\src\\..\\assets\\style.css"), Asset("css", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\Interact\\PENUy\\src\\..\\assets\\bulma_confined.min.css")], Dict{Any, Any}("changes" => Any[WebIO.JSString("(function (val){return (val!=this.model[\"changes\"]()) ? (this.valueFromJulia[\"changes\"]=true, this.model[\"changes\"](val)) : undefined})")], "index" => Any[WebIO.JSString("(function (val){return (val!=this.model[\"index\"]()) ? (this.valueFromJulia[\"index\"]=true, this.model[\"index\"](val)) : undefined})")]), WebIO.ConnectionPool(Channel{Any}(32), Set{AbstractConnection}(), Base.GenericCondition(ReentrantLock())), WebIO.JSString[WebIO.JSString("function () {\n    var handler = (function (ko, koPunches) {\n    ko.punches.enableAll();\n    ko.bindingHandlers.numericValue = {\n        init: function(element, valueAccessor, allBindings, data, context) {\n            var stringified = ko.observable(ko.unwrap(valueAccessor()));\n            stringified.subscribe(function(value) {\n                var val = parseFloat(value);\n                if (!isNaN(val)) {\n                    valueAccessor()(val);\n                }\n            });\n            valueAccessor().subscribe(function(value) {\n                var str = JSON.stringify(value);\n                if ((str == \"0\") && ([\"-0\", \"-0.\"].indexOf(stringified()) >= 0))\n                     return;\n                 if ([\"null\", \"\"].indexOf(str) >= 0)\n                     return;\n                stringified(str);\n            });\n            ko.applyBindingsToNode(\n                element,\n                {\n                    value: stringified,\n                    valueUpdate: allBindings.get('valueUpdate'),\n                },\n                context,\n            );\n        }\n    };\n    var json_data = {\"formatted_vals\":[\"0.0\",\"0.05\",\"0.1\",\"0.15\",\"0.2\",\"0.25\",\"0.3\",\"0.35\",\"0.4\",\"0.45\",\"0.5\",\"0.55\",\"0.6\"],\"changes\":WebIO.getval({\"name\":\"changes\",\"scope\":\"4464288755322532535\",\"id\":\"34\",\"type\":\"observable\"}),\"index\":WebIO.getval({\"name\":\"index\",\"scope\":\"4464288755322532535\",\"id\":\"33\",\"type\":\"observable\"})};

## 4. El Sistema de Seguridad Social de Capitalización

En un sistema de Seguridad Social de capitalización, los trabajadores contribuyen obligatoriamente con una fracción $\tau^{ss}$ de su salario durante su periodo activo ($t < t^*$). Estas contribuciones se acumulan en un fondo de pensiones $D_t$ que obtiene rentabilidad a la tasa de interés de mercado $R$.

Al jubilarse ($t = t^*$), el individuo recibe el fondo acumulado como un pago único ($Pension$).

### 3.1 Sustitución Perfecta de Ahorro
En este modelo, el agente tiene previsión perfecta y libre acceso al mercado financiero (sin restricciones de liquidez). Por lo tanto, el ahorro forzoso de la Seguridad Social es un **sustituto perfecto** del ahorro privado voluntario.
Si el gobierno aumenta la tasa impositiva $\tau^{ss}$, el consumo óptimo del agente **no varía**. En su lugar, el agente reduce proporcionalmente su ahorro privado voluntario (llegando a ser negativo si es necesario tomar prestado contra el fondo de jubilación bloqueado) para mantener la misma trayectoria de consumo.


## 5. Cuaderno de Bitácora (Actividades para el Alumno)

Responde a las siguientes cuestiones tras interactuar con las simulaciones de política fiscal:

1.  **Análisis de Equivalencia Ricardiana**:
    *   Activa la casilla `Devolver recaudación` en la simulación 1. Cambia el impuesto $\tau^w$ de 0 a 0.60. ¿Qué le ocurre al consumo en el panel de la izquierda? Explica detalladamente en base a la teoría microeconómica por qué no cambia.
    *   Desactiva la casilla `Devolver recaudación`. ¿Cómo responde el consumo y el ahorro? ¿Por qué la pendiente temporal del consumo sigue siendo positiva y con el mismo crecimiento relativo?
2.  **Efectos del Menú Fiscal Distorsionador**:
    *   En la simulación 2, establece `Equilibrio G = Recaudación` en `False`. Incrementa el impuesto sobre el trabajo $\tau^w$ de 0.0 a 0.50. ¿Cómo reacciona la oferta de trabajo $L_t$? ¿Por qué?
    *   Establece ahora un impuesto al consumo $\tau^c$ de 0.30 manteniendo $\tau^w = 0.0$. ¿Qué diferencia observas en el impacto sobre la oferta de trabajo respecto a $\tau^w$? ¿A qué se debe esto?
    *   Incrementa el impuesto al capital $\tau^r$ a 0.60. Observa el gráfico de Consumo. ¿Por qué la trayectoria se vuelve más plana (horizontal)? Explica el efecto intertemporal de la pérdida de interés real.
3.  **Seguridad Social e Implicaciones de Ahorro**:
    *   En la simulación 3, aumenta la cotización obligatoria $\tau^{ss}$ de 0.0 a 0.45. ¿Cambia el consumo óptimo del agente?
    *   Describe qué le ocurre a los activos privados (Panel 2) durante el periodo de vida laboral activa (antes de $t^* = 26$). ¿Por qué los activos privados caen a niveles fuertemente negativos (deuda)? Explica la lógica de "sustitución de ahorro" bajo previsión perfecta.


## 7. Benchmark de Rendimiento (Fase III)
Evaluamos la velocidad de simulación usando `BenchmarkTools.jl`.

In [11]:
# @btime (BenchmarkTools.jl) ejecuta la función repetidamente y muestra el
# tiempo mínimo y la memoria asignada. Los "$" evitan que las variables se
# traten como globales (falsearía la medición). (Fase III: rendimiento.)
@btime solve_distortionary_foc($params_lumpsum, fill(30.0, $params_lumpsum.T), true)


  106.100 μs (8965 allocations: 166.80 KiB)


Dict{String, Vector{Float64}} with 5 entries:
  "B"   => [-4.32419, -8.37102, -12.1388, -15.6258, -18.8301, -21.7499, -24.383…
  "C"   => [17.1621, 16.9802, 16.8002, 16.6221, 16.4459, 16.2716, 16.0991, 15.9…
  "L"   => [0.42793, 0.433994, 0.439994, 0.44593, 0.451803, 0.457614, 0.463363,…
  "W_L" => [12.8379, 13.0198, 13.1998, 13.3779, 13.5541, 13.7284, 13.9009, 14.0…
  "O"   => [0.57207, 0.566006, 0.560006, 0.55407, 0.548197, 0.542386, 0.536637,…